# K03_00 – Datenrepräsentation und Vorverarbeitung – Dozentenversion

Diese Fassung enthält **Musterlösungen und kurze didaktische Hinweise**.

Dieses Notebook ist der **Einstieg in Kapitel 3**.  
Wir schauen uns an, wie tabellarische Daten im Machine Learning dargestellt werden und warum Vorverarbeitung oft notwendig ist.

## Lernziele
Nach diesem Notebook können Sie:
- zwischen **Samples**, **Features** und **Label** unterscheiden
- numerische und kategoriale Merkmale erkennen
- **fehlende Werte** identifizieren
- einfache **One-Hot-Kodierung** anwenden
- den Effekt von **Skalierung** verstehen

## 1. Ein kleines Beispieldataset

Wir verwenden einen kleinen fiktiven Datensatz zu Wohnungsangeboten.  
Jede **Zeile** ist ein Sample, jede **Spalte** ein Merkmal.

Die Zielvariable heißt hier `preis_in_tsd`.


In [1]:
import pandas as pd
import numpy as np

df = pd.DataFrame({
    "wohnflaeche": [45, 52, 68, 80, 95, 110, 130, 70],
    "zimmer": [2, 2, 3, 3, 4, 4, 5, 3],
    "lage": ["einfach", "mittel", "mittel", "gut", "gut", "gut", "sehr_gut", "mittel"],
    "baujahr": [1970, 1985, 1995, 2001, np.nan, 2010, 2018, 1998],
    "preis_in_tsd": [120, 145, 180, 220, 260, 310, 420, 195]
})

df


,wohnflaeche,zimmer,lage,baujahr,preis_in_tsd
0,45,2,einfach,1970.0,120
1,52,2,mittel,1985.0,145
2,68,3,mittel,1995.0,180
3,80,3,gut,2001.0,220
4,95,4,gut,NaN,260
5,110,4,gut,2010.0,310
6,130,5,sehr_gut,2018.0,420
7,70,3,mittel,1998.0,195


### Merksatz
- **Sample** = eine Beobachtung, also eine Zeile
- **Features** = beschreibende Merkmale
- **Label** = Zielwert, den wir vorhersagen wollen


In [2]:
X = df.drop(columns="preis_in_tsd")
y = df["preis_in_tsd"]

print("Feature-Matrix X:")
display(X)

print("Zielvariable y:")
display(y)


Feature-Matrix X:


,wohnflaeche,zimmer,lage,baujahr
0,45,2,einfach,1970.0
1,52,2,mittel,1985.0
2,68,3,mittel,1995.0
3,80,3,gut,2001.0
4,95,4,gut,NaN
5,110,4,gut,2010.0
6,130,5,sehr_gut,2018.0
7,70,3,mittel,1998.0


Zielvariable y:


,preis_in_tsd
0,120
1,145
2,180
3,220
4,260
5,310
6,420
7,195


## Mini-Übung 1
Beantworten Sie kurz:
1. Wie viele Samples enthält der Datensatz?
2. Wie viele Features hat `X`?
3. Welche Spalte ist das Label?


### Musterlösung / Dozentenhinweis
1. Der Datensatz enthält **8 Samples**.
2. `X` enthält **3 Features**: `wohnflaeche`, `zimmer`, `baujahr`.
3. Das Label ist die Spalte **`preis_in_tsd`**.

Didaktischer Hinweis: Hier lohnt es sich, die Begriffe **Zeile = Sample**, **Spalte = Feature**, **Zielspalte = Label** nochmals explizit mit der Tabelle zu verknüpfen.

In [3]:
print("Anzahl Samples:", X.shape[0])
print("Anzahl Features:", X.shape[1])
print("Label-Spalte:", "preis_in_tsd")


Anzahl Samples: 8
Anzahl Features: 4
Label-Spalte: preis_in_tsd


## 2. Numerische und kategoriale Merkmale

Nicht alle Spalten haben denselben Typ:
- `wohnflaeche`, `zimmer`, `baujahr` sind **numerisch**
- `lage` ist **kategorial**

Viele klassische ML-Modelle erwarten am Ende **numerische Eingaben**.


In [4]:
print(df.dtypes)


wohnflaeche       int64
zimmer            int64
lage             object
baujahr         float64
preis_in_tsd      int64
dtype: object


## 3. Fehlende Werte

In realen Daten sind Einträge oft unvollständig.  
Hier fehlt ein Wert in der Spalte `baujahr`.


In [5]:
print(df.isna().sum())


wohnflaeche     0
zimmer          0
lage            0
baujahr         1
preis_in_tsd    0
dtype: int64


### Mini-Übung 2
Welche Strategien wären hier denkbar?
- Zeile löschen
- Spalte löschen
- Wert schätzen (imputieren)

Im nächsten Schritt verwenden wir eine einfache Schätzung mit dem Median.


### Musterlösung / Dozentenhinweis
Mögliche Strategien sind tatsächlich:
- **Zeile löschen**, wenn nur sehr wenige Fälle betroffen sind
- **Spalte löschen**, wenn sehr viele Werte fehlen oder das Merkmal kaum relevant ist
- **Wert imputieren**, z. B. mit Mittelwert, Median oder häufigstem Wert

Hier ist der **Median** didaktisch sinnvoll, weil er robuster gegenüber Ausreißern ist als der Mittelwert.

In [6]:
# Zuerst wird eine Kopie des originalen DataFrames df erstellt und df_imputed genannt.
df_imputed = df.copy()

# Als Nächstes wird der Medianwert der Spalte 'baujahr' berechnet.
# Der Median ist eine robuste Wahl zum Ersetzen fehlender Werte, da er weniger
# empfindlich auf Ausreißer reagiert als der Mittelwert.
median_baujahr = df_imputed["baujahr"].median()

# In diesem Schritt werden alle fehlenden Werte (NaN) in der Spalte 'baujahr'
# des df_imputed-DataFrames durch den zuvor berechneten Medianwert ersetzt.
df_imputed["baujahr"] = df_imputed["baujahr"].fillna(median_baujahr)

print("Verwendeter Median:", median_baujahr)
df_imputed


Verwendeter Median: 1998.0


,wohnflaeche,zimmer,lage,baujahr,preis_in_tsd
0,45,2,einfach,1970.0,120
1,52,2,mittel,1985.0,145
2,68,3,mittel,1995.0,180
3,80,3,gut,2001.0,220
4,95,4,gut,1998.0,260
5,110,4,gut,2010.0,310
6,130,5,sehr_gut,2018.0,420
7,70,3,mittel,1998.0,195


## 4. Kategoriale Merkmale kodieren

Die Spalte `lage` ist Text.  
Damit ein Modell später damit arbeiten kann, kodieren wir sie mit **One-Hot-Encoding**.


In [7]:
# pd.get_dummies() ist eine Funktion aus der Pandas-Bibliothek, die verwendet
# wird, um kategoriale (textbasierte) Daten in ein numerisches Format
# umzuwandeln, das maschinelles Lernmodelle verarbeiten können.
df_encoded = pd.get_dummies(df_imputed, columns=["lage"], dtype=int)

df_encoded


,wohnflaeche,zimmer,baujahr,preis_in_tsd,lage_einfach,lage_gut,lage_mittel,lage_sehr_gut
0,45,2,1970.0,120,1,0,0,0
1,52,2,1985.0,145,0,0,1,0
2,68,3,1995.0,180,0,0,1,0
3,80,3,2001.0,220,0,1,0,0
4,95,4,1998.0,260,0,1,0,0
5,110,4,2010.0,310,0,1,0,0
6,130,5,2018.0,420,0,0,0,1
7,70,3,1998.0,195,0,0,1,0


### Beobachtung
Aus **einer** kategorialen Spalte entstehen **mehrere** numerische Spalten.


## 5. Warum ist Skalierung wichtig?

Betrachten wir zwei Merkmale:
- `wohnflaeche` liegt ungefähr zwischen 45 und 130
- `zimmer` liegt nur zwischen 2 und 5

Ohne Skalierung hat `wohnflaeche` numerisch eine viel größere Größenordnung.
Das kann für distanzbasierte Verfahren problematisch sein.


In [8]:
from sklearn.preprocessing import StandardScaler, MinMaxScaler

features_numeric = df_imputed[["wohnflaeche", "zimmer", "baujahr"]]

standard_scaler = StandardScaler()
minmax_scaler = MinMaxScaler()

X_standard = pd.DataFrame(
    standard_scaler.fit_transform(features_numeric),
    columns=features_numeric.columns
)

X_minmax = pd.DataFrame(
    minmax_scaler.fit_transform(features_numeric),
    columns=features_numeric.columns
)

print("Originaldaten:")
display(features_numeric.head())

print("Standardisiert:")
display(X_standard.head())

print("Min-Max-skaliert:")
display(X_minmax.head())


Originaldaten:


,wohnflaeche,zimmer,baujahr
0,45,2,1970.0
1,52,2,1985.0
2,68,3,1995.0
3,80,3,2001.0
4,95,4,1998.0


Standardisiert:


,wohnflaeche,zimmer,baujahr
0,-1.338753,-1.290994,-1.959491
1,-1.080235,-1.290994,-0.865822
2,-0.489337,-0.258199,-0.136709
3,-0.046164,-0.258199,0.300759
4,0.507803,0.774597,0.082025


Min-Max-skaliert:


,wohnflaeche,zimmer,baujahr
0,0.000000,0.000000,0.000000
1,0.082353,0.000000,0.312500
2,0.270588,0.333333,0.520833
3,0.411765,0.333333,0.645833
4,0.588235,0.666667,0.583333


## Mini-Übung 3
Prüfen Sie:
1. Welchen Mittelwert hat jede Spalte nach der Standardisierung ungefähr?
2. In welchem Bereich liegen die Werte nach der Min-Max-Skalierung?


### Musterlösung / Dozentenhinweis
1. Nach der **Standardisierung** haben die Spalten  einen Mittelwert von mathematisch exakt **0**.
2. Nach der **Min-Max-Skalierung** liegen die Werte im Bereich **[0, 1]**.

Wichtiger Merksatz: Standardisierung und Min-Max-Skalierung verfolgen unterschiedliche Ziele; beide machen Merkmale aber besser vergleichbar.

**StandardScaler** wenn:

die Daten normalverteilt sind (Glockenkurve)

es Ausreißer gibt (StandardScaler ist robuster dagegen)

du Algorithmen verwendest die Normalverteilung voraussetzen (z.B. Logistische Regression, SVM, neuronale Netze)

**Min-Max-Skalierung**  wenn:

du Werte garantiert zwischen 0 und 1 brauchst

die Daten keine starken Ausreißer haben

du Bilder verarbeitest (Pixelwerte 0-255 → 0-1 ist Standard)

du neuronale Netze mit Sigmoid-Aktivierung verwendest (erwartet Werte zwischen 0 und 1)

In [ ]:
print("Mittelwerte nach Standardisierung:")
print(X_standard.mean().round(6))

print("\nMinima nach Min-Max-Skalierung:")
print(X_minmax.min().round(6))

print("\nMaxima nach Min-Max-Skalierung:")
print(X_minmax.max().round(6))


## 6. Kleine Zusammenfassung

In diesem Notebook haben wir gesehen:
- tabellarische Daten werden in **X** und **y** getrennt
- Merkmale können unterschiedliche Typen haben
- fehlende Werte müssen behandelt werden
- kategoriale Merkmale müssen oft kodiert werden
- Skalierung ist für viele Verfahren methodisch wichtig

